# Orderbook Store Example

This notebook demonstrates the new data storage infrastructure for OKX orderbook data.


In [1]:
%load_ext autoreload
%autoreload 2

from datetime import datetime, date
from okx.store import OrderbookStore
import polars as pl


## 1. Setup Store


In [2]:
# Initialize the store
store = OrderbookStore(
    data_root="data/okx",
    manifest_path="data/okx/manifest.sqlite"
)


## 2. Populate Raw Data

Fetch and store full depth=5 orderbook data. This only fetches missing dates.


In [5]:
# Populate BTC-USD-SWAP data for a few days
store.populate(
    inst_type='SWAP',
    inst_family='BTC-USD',
    start=datetime(2025, 8, 1),
    end=datetime(2025, 9, 1),
    verbose=True
)


Fetching BTC-USD/SWAP: 2 missing dates (from 31 requested)


  2025-08-07: No data returned from API
  2025-08-09: No data returned from API
✓ Stored 0/2 days as raw orderbook data
  Failed dates: [datetime.date(2025, 8, 7), datetime.date(2025, 8, 9)]


## 3. Example Queries

### 3.1 Depth=0, 1-minute bins (with caching)

When depth=0, top orderbook levels are not available. Instead, only a mid price is kept. By default, depth=0 also deletes top level qty and and ordCnt data.


In [9]:
# Get depth=0, 1-min binned data (cached as 'd0_1m')
lf = store.get(
    inst_type='FUTURES',
    inst_family='BTC-USD',
    start=datetime(2025, 9, 1),
    end=datetime(2025, 9, 5),
    depth=0,
    binning='1m',
    cache_name='d0_1m'
)

# Collect and display
df = lf.collect()
print(f"Shape: {df.shape}")
df.head()


Shape: (40320, 5)


symbol,timeMs,exchTimeMs,mid,time_bin
str,i64,i64,f64,datetime[ms]
"""BTC-USD-250905.OK""",1756684859478,1756684859476,108314.45,2025-09-01 00:01:00
"""BTC-USD-250905.OK""",1756684919757,1756684919756,108382.35,2025-09-01 00:02:00
"""BTC-USD-250905.OK""",1756684979088,1756684979086,108308.15,2025-09-01 00:03:00
"""BTC-USD-250905.OK""",1756685039387,1756685039386,108275.05,2025-09-01 00:04:00
"""BTC-USD-250905.OK""",1756685099938,1756685099936,108146.05,2025-09-01 00:05:00


### 3.2 Depth=1, 5-minute bins with features (with caching)

Default behavior applies features in order: features → trim → bin.
Trim and bin are not applied at the beginning by default to ensure full orderbook availability, as features may need information that's not available after trim/bin.

However, it's best practice to apply trim and bin as early as possible to improve efficiency. In this example, it is safe to place them before the other features.

In [8]:
# Get depth=1, 5-min bins with mid, spread, and imbalance features
lf = store.get(
    inst_type='SWAP',
    inst_family='BTC-USD',
    start=datetime(2025, 9, 1),
    end=datetime(2025, 9, 5),
    depth=1,
    binning='5m',
    features=['trim', 'bin', 'mid', 'spread', 'rel_spread', 'imbalance1'],
    cache_name='d1_5m_features'
)

df = lf.collect()
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns}")
df.head()


Shape: (1152, 14)
Columns: ['symbol', 'timeMs', 'exchTimeMs', 'bid_1_px', 'bid_1_qty', 'bid_1_ordCnt', 'ask_1_px', 'ask_1_qty', 'ask_1_ordCnt', 'time_bin', 'mid', 'spread', 'rel_spread', 'imbalance1']


symbol,timeMs,exchTimeMs,bid_1_px,bid_1_qty,bid_1_ordCnt,ask_1_px,ask_1_qty,ask_1_ordCnt,time_bin,mid,spread,rel_spread,imbalance1
str,i64,i64,f64,f64,i32,f64,f64,i32,datetime[ms],f64,f64,f64,f64
"""BTC-USD-SWAP.OK""",1756685099998,1756685099996,108052.9,4313.0,28,108053.0,3569.0,11,2025-09-01 00:05:00,108052.95,0.1,9.2547e-7,0.094392
"""BTC-USD-SWAP.OK""",1756685399987,1756685399985,107941.7,143.0,7,107941.8,4960.0,28,2025-09-01 00:10:00,107941.75,0.1,9.2643e-7,-0.943955
"""BTC-USD-SWAP.OK""",1756685699986,1756685699985,107924.0,989.0,12,107924.1,6369.0,28,2025-09-01 00:15:00,107924.05,0.1,9.2658e-7,-0.731177
"""BTC-USD-SWAP.OK""",1756685999986,1756685999985,107898.0,3893.0,20,107898.1,368.0,5,2025-09-01 00:20:00,107898.05,0.1,9.2680e-7,0.827271
"""BTC-USD-SWAP.OK""",1756686299997,1756686299995,107697.7,2922.0,36,107697.8,170.0,2,2025-09-01 00:25:00,107697.75,0.1,9.2852e-7,0.890039


### 3.3 Custom features with lambda function

In many cases, the qty and ordCnt columns are not necessary, as well as exchTime. Placing 'strip' early in the feature list may be useful, as it keeps only 'symbol', 'timeMs', and any existing price columns (_px or mid)

In [11]:
# Custom feature: compute volatility over rolling window
def add_volatility(lf: pl.LazyFrame) -> pl.LazyFrame:
    return lf.with_columns([
        pl.col('mid').rolling_std(window_size=10).alias('mid_vol_10')
    ])

# Combine registry features with custom function
# Applying bin and trim first - not default behavoir but desired for volatility.
lf = store.get(
    inst_type='SWAP',
    inst_family='BTC-USD',
    start=datetime(2025, 9, 1),
    end=datetime(2025, 9, 5),
    depth=1,
    binning='1m',
    features=['trim', 'strip', 'bin', 'mid', 'spread', add_volatility],
    cache_name='d1_1m_vol'
)

df = lf.collect()
print(f"Columns: {df.columns}")
df.head(15)


Columns: ['symbol', 'timeMs', 'bid_1_px', 'ask_1_px', 'time_bin', 'mid', 'spread', 'mid_vol_10']


symbol,timeMs,bid_1_px,ask_1_px,time_bin,mid,spread,mid_vol_10
str,i64,f64,f64,datetime[ms],f64,f64,f64
"""BTC-USD-SWAP.OK""",1756684859827,108213.9,108214.0,2025-09-01 00:01:00,108213.95,0.1,null
"""BTC-USD-SWAP.OK""",1756684919998,108288.1,108288.2,2025-09-01 00:02:00,108288.15,0.1,null
"""BTC-USD-SWAP.OK""",1756684979998,108207.1,108207.2,2025-09-01 00:03:00,108207.15,0.1,null
"""BTC-USD-SWAP.OK""",1756685039857,108180.0,108180.1,2025-09-01 00:04:00,108180.05,0.1,null
"""BTC-USD-SWAP.OK""",1756685099998,108052.9,108053.0,2025-09-01 00:05:00,108052.95,0.1,null
…,…,…,…,…,…,…,…
"""BTC-USD-SWAP.OK""",1756685459997,107963.8,107963.9,2025-09-01 00:11:00,107963.85,0.1,113.44479
"""BTC-USD-SWAP.OK""",1756685519998,107972.1,107972.2,2025-09-01 00:12:00,107972.15,0.1,91.033061
"""BTC-USD-SWAP.OK""",1756685579998,107965.9,107966.0,2025-09-01 00:13:00,107965.95,0.1,75.051624


### 3.4 Ad-hoc query (no caching)


In [13]:
# Quick query without caching - just compute on the fly
lf = store.get(
    inst_type='SWAP',
    inst_family='BTC-USD',
    start=datetime(2025, 9, 1),
    end=datetime(2025, 9, 5),
    depth=2,
    binning='10m'
)

df = lf.collect()
print(f"Columns: {df.columns}")
print(f"Shape: {df.shape}")
df.head()


Columns: ['symbol', 'timeMs', 'exchTimeMs', 'bid_1_px', 'bid_1_qty', 'bid_1_ordCnt', 'ask_1_px', 'ask_1_qty', 'ask_1_ordCnt', 'bid_2_px', 'bid_2_qty', 'bid_2_ordCnt', 'ask_2_px', 'ask_2_qty', 'ask_2_ordCnt', 'time_bin']
Shape: (576, 16)


symbol,timeMs,exchTimeMs,bid_1_px,bid_1_qty,bid_1_ordCnt,ask_1_px,ask_1_qty,ask_1_ordCnt,bid_2_px,bid_2_qty,bid_2_ordCnt,ask_2_px,ask_2_qty,ask_2_ordCnt,time_bin
str,i64,i64,f64,f64,i32,f64,f64,i32,f64,f64,i32,f64,f64,i32,datetime[ms]
"""BTC-USD-SWAP.OK""",1756685399987,1756685399985,107941.7,143.0,7,107941.8,4960.0,28,107939.1,2.0,1,107941.9,128.0,2,2025-09-01 00:10:00
"""BTC-USD-SWAP.OK""",1756685999986,1756685999985,107898.0,3893.0,20,107898.1,368.0,5,107896.0,48.0,1,107901.9,15.0,1,2025-09-01 00:20:00
"""BTC-USD-SWAP.OK""",1756686599996,1756686599995,107766.0,1702.0,12,107766.1,2075.0,19,107764.3,2.0,1,107768.7,207.0,1,2025-09-01 00:30:00
"""BTC-USD-SWAP.OK""",1756687199946,1756687199945,107713.9,3390.0,34,107714.0,89.0,2,107713.8,15.0,1,107716.0,48.0,1,2025-09-01 00:40:00
"""BTC-USD-SWAP.OK""",1756687799997,1756687799995,108127.9,4730.0,29,108128.0,593.0,4,108126.0,48.0,1,108128.9,250.0,5,2025-09-01 00:50:00


## 4. Cache Management


In [3]:
# List all available caches
print("Available caches:")
for cache in store.manifest.list_caches():
    print(f"  - {cache}")


Available caches:
  - 30s_pillars
  - 1m_pillars
  - 5m_pillars
  - full_pillars
  - 30s_options
  - 1m_options
  - 5m_options
  - full_options
  - 30s_kalman_forwards
  - 1m_kalman_forwards
  - 1m_pchip_forwards
  - 5m_kalman_forwards
  - deep_eval_kalman_1m
  - deep_eval_pchip_1m
  - deep_eval_pchip_full
  - full_kalman_forwards
  - full_pchip_forwards


In [6]:
# Clear a specific cache
store.clear_cache('30s_kalman_forwards')


Cleared cache: 30s_kalman_forwards


In [18]:
# Clear all caches (keeps raw data intact)
store.clear_cache()


Cleared all caches


Raw data can also be deleted if desired:

In [ ]:
# Let's delete one day's data for BTC-USD FUTURES (e.g. August 15, 2025)
delete_date = date(2025, 8, 15)

# store.delete_raw('BTC-USD', 'FUTURES', delete_date) (uncomment to delete)

# You can check if it's deleted from the manifest:
have_after = store.manifest.have('BTC-USD', 'FUTURES', delete_date, 'raw')
print(f"Raw for {delete_date} present after delete? {have_after}")